# Review all six canonical variables in one run

This batch workflow is separate from `1.variable_decide.ipynb`, which still
supports reviewing one variable. Run sections 1–3 **once** to check all six
variables and save all plots and readouts. Inspect the outputs, edit the manual
choices in section 4, and run section 5 to save final rules. Editing manual choices
requires rerunning only sections 4–5, with no database queries or plotting.


## 1. Setup and settings


In [1]:
%load_ext autoreload
%autoreload 2

from pathlib import Path
import os
import sys
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import sqlalchemy as sa
from IPython.display import display

HERE = Path.cwd()
if not (HERE / 'func' / 'func_variable_mapping.py').exists():
    HERE = HERE / 'Quanlity_Control' / 'variable_mapping'
if not (HERE / 'func' / 'func_variable_mapping.py').exists():
    raise FileNotFoundError('Run from the repository root or variable_mapping folder.')
sys.path.insert(0, str(HERE / 'func'))
from func_variable_mapping import CANONICAL_VARIABLES
from func_variable_review import (
    candidate_source_time_scales, format_candidate_review,
    run_variable_reviews, save_review_rules,
)

DB_URL = os.getenv(
    'CRMP_DB_URL',
    'postgresql://tongli1997@proddb01.pcic.uvic.ca,proddb02.pcic.uvic.ca/crmp'
    '?keepalives=1&keepalives_idle=300&keepalives_interval=300'
    '&keepalives_count=9&passfile=/workspaces/crmprtd/.pgpass',
)
engine = sa.create_engine(DB_URL, pool_pre_ping=True)
plt.style.use('seaborn-v0_8-whitegrid')
pd.set_option('display.max_columns', 50)

In [2]:
NETWORK_ID = 76  # Change only this value to work on another network.
NETWORK_DIR = HERE / 'networks' / f'network_{NETWORK_ID}'
NETWORK_DIR.mkdir(parents=True, exist_ok=True)
VARIABLES_TO_REVIEW = list(CANONICAL_VARIABLES)  # All six in one run.
PLOT_STATION_LIMIT = 10
SOURCE_STATION_LIMIT = 20
RANDOM_SEED = 42  # None for a fresh sample; integer for repeatable samples.
AUTO_KEEP_FROM = '2026-01-01'
OUTPUT_PREFIX = f'net{NETWORK_ID}_'
REGISTRY_PATH = NETWORK_DIR / f'{OUTPUT_PREFIX}candidate_mappings.csv'
FINAL_RULE_PATH = NETWORK_DIR / f'{OUTPUT_PREFIX}final_variable_rules.csv'

## 2. Run all variables once

For each canonical variable, the runner finds stations containing **every candidate
ID**, then samples up to 20 of those stations for recent-data checks and up to 10
for plots. The two samples are drawn separately; a fixed seed can make them overlap.
Eligibility requires observations at any date, including null values; timestamps
need not match between candidates. The recent-data check requires non-null raw
observations **on or after January 1, 2026**, before plausible-value filtering.

Automatic keeps, `recent_non_null_station_count`, and latest observation dates describe
the checked sample, not the entire network. `network_station_count` is network-wide.
Eligible-station discovery still checks the network,
but is done only once per variable; recent-data queries use only the sampled stations.
The runner first looks for stations containing every candidate. If none exist, it
automatically falls back to stations containing any candidate; the prefixed run-summary CSV
records the selection method. A variable with missing candidates, no stations, or no
plausible plot data is reported there. Unexpected database errors stop the run.

Each successful run replaces `networks/network_<id>/latest_review/` for the selected
network. Results for other networks are preserved. If a run fails, the previous
successful `latest_review/` remains available:

- `net<N>_time_series.pdf`: all variables, one station per page.
- `net<N>_cdf.pdf`: all variables, one station per page. Precipitation and snowfall use
  positive values on a logarithmic axis; other variables use all plausible values.
- `net<N>_candidate_review.csv`: network-wide station counts, sampled recent-data counts, latest sample dates, and automatic decisions. `network_station_count` counts distinct stations with at least one non-null observation at any date; `recent_non_null_station_count` applies only to sampled stations on or after the cutoff.
- `net<N>_station_conditions.csv`: per-station recent-data conditions, organized by variable ID.
- `net<N>_qc_summary.csv`: plausible-value filtering readouts.
- `net<N>_run_summary.csv`: status and station selection for each canonical variable.
- `net<N>_review_results.json`: saved results needed to apply manual choices later.

PDF pages include the canonical variable, station name/ID, and candidate ID legends.
Every plotted candidate legend is marked `daily` or `hourly` from the shared frequency registry.
All time-series pages for one canonical variable use the same full time range.
A missing-data page explains when a variable has no plots. No final rules are saved yet.


In [3]:
candidate_mappings = pd.read_csv(REGISTRY_PATH)
SOURCE_VARS_TIME_SCALES = candidate_source_time_scales(
    candidate_mappings, NETWORK_ID)
batch_results = run_variable_reviews(
    engine, candidate_mappings, NETWORK_DIR,
    network_id=NETWORK_ID,
    variables=VARIABLES_TO_REVIEW,
    plot_station_limit=PLOT_STATION_LIMIT,
    source_station_limit=SOURCE_STATION_LIMIT,
    random_seed=RANDOM_SEED,
    auto_keep_from=AUTO_KEEP_FROM,
    source_vars_time_scales=SOURCE_VARS_TIME_SCALES,
)
RUN_DIR = Path(batch_results['run_dir'])

Reviewing air_temperature...
Reviewing daily_min_temperature...
Reviewing daily_max_temperature...
Reviewing precipitation_amount...
Reviewing snowfall_amount...
Reviewing snow_depth...
Review complete: /workspaces/crmprtd/Quanlity_Control/variable_mapping/networks/network_76/latest_review


## 3. Inspect saved readouts and figures

This cell only reads saved files; rerunning it does not query the database.
Counts and latest dates apply to the source-check sample. `source_checked_at_utc`
is the time that check finished. Empty latest dates mean no qualifying observation
was found in the sample, which does not prove absence elsewhere in the network.


In [4]:

from IPython.display import FileLink

display(pd.read_csv(RUN_DIR / f'{OUTPUT_PREFIX}run_summary.csv'))
source_review = format_candidate_review(
    pd.read_csv(RUN_DIR / f'{OUTPUT_PREFIX}candidate_review.csv'))
display(source_review.sort_values(['canonical_variable', 'vars_id']).set_index(
    ['canonical_variable', 'vars_id']
))
for filename in [f'{OUTPUT_PREFIX}time_series.pdf', f'{OUTPUT_PREFIX}cdf.pdf',
                 f'{OUTPUT_PREFIX}candidate_review.csv',
                 f'{OUTPUT_PREFIX}station_conditions.csv',
                 f'{OUTPUT_PREFIX}qc_summary.csv', f'{OUTPUT_PREFIX}run_summary.csv']:
    display(FileLink(str((RUN_DIR / filename).relative_to(Path.cwd()))
                     if (RUN_DIR / filename).is_relative_to(Path.cwd())
                     else str(RUN_DIR / filename)))

,canonical_variable,status,eligible_station_count,source_station_count,plot_station_ids,station_selection_method
0,air_temperature,ok,1,1,[15020],all_candidates
1,daily_min_temperature,no_candidates,0,0,[],not_applicable
2,daily_max_temperature,no_candidates,0,0,[],not_applicable
3,precipitation_amount,no_candidates,0,0,[],not_applicable
4,snowfall_amount,no_candidates,0,0,[],not_applicable
5,snow_depth,no_candidates,0,0,[],not_applicable


network_id         net_var_name  \
canonical_variable vars_id                                    
air_temperature    1295             76            TEMP_MEAN   
                   1297             76  avg_air_temp_pst1hr   

                            network_station_count source_time_scale  \
canonical_variable vars_id                                            
air_temperature    1295                         1            hourly   
                   1297                         1            hourly   

                           daily_aggregation         decision   source_family  \
canonical_variable vars_id                                                      
air_temperature    1295                 mean  manual decision  network76_auto   
                   1297                 mean  manual decision  network76_auto   

                            recent_non_null_station_count  \
canonical_variable vars_id                                  
air_temperature    1295                                 0   
                   1297                                 0   

                                       source_checked_at_utc  \
canonical_variable vars_id                                     
air_temperature    1295     2026-11-16T22:23:07.336436+00:00   
                   1297     2026-11-16T22:23:07.336436+00:00   

                            latest_observation_from_cutoff  priority  
canonical_variable vars_id                                            
air_temperature    1295                                NaN         1  
                   1297                                NaN         2

/workspaces/crmprtd/Quanlity_Control/variable_mapping/networks/network_76/latest_review/net76_time_series.pdf

/workspaces/crmprtd/Quanlity_Control/variable_mapping/networks/network_76/latest_review/net76_cdf.pdf

/workspaces/crmprtd/Quanlity_Control/variable_mapping/networks/network_76/latest_review/net76_candidate_review.csv

/workspaces/crmprtd/Quanlity_Control/variable_mapping/networks/network_76/latest_review/net76_station_conditions.csv

/workspaces/crmprtd/Quanlity_Control/variable_mapping/networks/network_76/latest_review/net76_qc_summary.csv

/workspaces/crmprtd/Quanlity_Control/variable_mapping/networks/network_76/latest_review/net76_run_summary.csv

## 4. Edit manual source IDs after reviewing the outputs

`MANUAL_SOURCE_IDS_BY_NETWORK` is the durable registry of manually retained source IDs.
Each network key maps canonical variables to reviewed candidate IDs. Update this one
cell when a manual decision changes, and use only IDs listed in that network's candidate
review. Networks with no manual selections may be omitted. Automatically recognized and
single-candidate selections do not need to be repeated here. All choices are validated
before saving, and existing final source selections remain unchanged on an ordinary rerun.


In [5]:
# Persist reviewed additions by network so changing NETWORK_ID is safe.
MANUAL_SOURCE_IDS_BY_NETWORK = {
    1: {
        'precipitation_amount': [429],
    },
    2: {
        'air_temperature': [434],
        'daily_min_temperature': [435],
        'daily_max_temperature': [433],
        'precipitation_amount': [442],
        'snowfall_amount': [451],
        'snow_depth': [450],
    },
    5: {
        'air_temperature': [1328, 1329],
        'snow_depth': [468, 1331],
    },
    6: {
        'daily_min_temperature': [469],
        'daily_max_temperature': [470],
        'precipitation_amount': [471],
    },
    9: {
        'air_temperature': [472, 693],
    },
    14: {
        'air_temperature': [701, 702],
        'precipitation_amount': [513],
    },
    17: {
        'precipitation_amount': [519],
    },
    70: {
        'air_temperature': [1305, 1306],
    },
    76: {
        'air_temperature': [1295, 1297],
    },
}
MANUAL_SOURCE_IDS = MANUAL_SOURCE_IDS_BY_NETWORK.get(NETWORK_ID, {})

## 5. Save final rules without rerunning the review

The save function combines automatic and manual IDs and updates one row per network
and canonical variable in the existing rules CSV. Other rules are preserved. Variables
with no selected IDs leave their existing rules unchanged. The same compact columns as
the single-variable workflow are used: sampled station IDs, the single check timestamp,
all candidate IDs, recent source IDs/latest dates, and merge policy. Detailed
conditions and counts remain
in separate readouts. The prefixed final-variable-rules CSV in this run's folder records the saved selections.

After a kernel restart, rerun sections 1–3 before editing section 4 and saving. Only
the latest successful review is retained for each network, so this notebook does not
rely on previous-run folders. Saving itself does not connect to the database.


In [6]:
final_rules = save_review_rules(
    batch_results, MANUAL_SOURCE_IDS, FINAL_RULE_PATH)
display(final_rules)
if FINAL_RULE_PATH.exists():
    print(f'Rules file: {FINAL_RULE_PATH}')
run_rules_path = RUN_DIR / FINAL_RULE_PATH.name
if run_rules_path.exists():
    print(f'Run-specific selections: {run_rules_path}')
else:
    print('No rules updated: no source IDs were selected for this run.')

,network_id,canonical_variable,source_vars_ids,source_vars_time_scales,candidate_vars_ids,recent_source_vars_ids,recent_source_latest_observations,auto_keep_from,source_check_scope,source_station_limit,source_random_seed,source_station_count,source_station_ids,source_checked_at_utc,time_match,overlap_method,single_source_method
0,76,air_temperature,"[1295, 1297]","{""1295"": ""hourly"", ""1297"": ""hourly""}","[1295, 1297]",[],{},2026-01-01,random_stations_with_all_candidates,20,42,1,[15020],2026-11-16T22:23:07.336436+00:00,exact_timestamp,mean,use_available


Rules file: /workspaces/crmprtd/Quanlity_Control/variable_mapping/networks/network_76/net76_final_variable_rules.csv
Run-specific selections: /workspaces/crmprtd/Quanlity_Control/variable_mapping/networks/network_76/latest_review/net76_final_variable_rules.csv
